In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.Connection('data.db')
cur = conn.cursor()

try:
    cur.execute('create index if not exists idx_suggestions_article_id on suggestions(article_id);')
    cur.execute('create index if not exists idx_suggestions_count on suggestions(count);')
    conn.commit()

finally:
    conn.close()

In [12]:
import pandas as pd
import sqlite3

pd.read_sql('''
            select 
			    *
            from embeddings
            order by random()
            limit 500
''', sqlite3.Connection('data.db'))

,id,created,modified,sentence_id,summary_id,article_id,embedding
0,2466066,1771675303349,1771675303349,None,66,None,b'(\x81g\xbb\xfef\xc4\xbb<$6\xbc\x80\xa8\x11=\...
1,2182522,1771671594020,1771671594020,None,522,None,b'\x1as\xeb\xbc\x07\x9d\xc6<\xa4Q@\xbc\xde@\xd...
2,1292864,1771660010110,1771660010110,None,864,None,b'\xb2\xf1^\xbd\xd2\x18|=-\xb6&=\x91\x02q\xbd\...
3,2669537,1771677941691,1771677941691,None,537,None,b'\xafY\xc1<\x99P\xb3;=\x8c\xc1<~\xba\x00=\x0b...
4,753060,1771652990000,1771652990000,None,60,None,b'\x01\x16:>\x83&k\xbcP{\xbd\xbc\x1e\xb1\xf3\x...
...,...,...,...,...,...,...,...
495,2424973,1771674757594,1771674757594,None,973,None,"b'\x93\x17\x9e\xbd=[8=\x92P.9\xc1\xa3/=""g\xff\..."
496,3147435,1771684173074,1771684173074,None,435,None,b'w.\xe0\xbd\xf6\x9a5\xbd\xba\x07\xe8\xbc&|\x1...
497,2847476,1771680261925,1771680261925,None,476,None,b'\xbc\x96\xba=l\xff\xed\xbd:~\x99=e\xa0\xa9<\...
498,437319,1771648885354,1771648885354,None,319,None,b'\xea\x9f\xb4<\xf1\t`\xbdds\x96=\x9e\xd2O=\x9...


In [20]:
import pandas as pd
import sqlite3

df = pd.read_sql('''
		select * from sentences join (
            select 
			    *
            from embeddings
            order by random()
            limit 500
        ) as sub on sub.sentence_id == sentences.id
''', sqlite3.Connection('data.db'))

df

,id,created,modified,sentence,tokens,article_id,id,created,modified,sentence_id,summary_id,article_id,embedding
0,763,1771560055275,1771560055275,A glass floor was installed on the first level...,42,Eiffel_Tower,1868763,1771667506999,1771667506999,763,763,None,b'\xc7\xb5\xf2<\x82O*=\xccl&\xbc\xfd\xc8\xaa=\...
1,984,1771560055741,1771560055741,The common ingredient in all types of sushi is...,40,Sushi,1871984,1771667545688,1771667545688,984,984,None,"b'\x92\xec\xbe\xbd\x8f,T\xbd\x8c^\\<`\xc9\x93=..."
2,113,1771560054124,1771560054124,The meaning of the name ice cream varies from ...,60,Ice_cream,1298113,1771660088405,1771660088405,113,113,None,b')\xa2\xc8<=\xf7\x8f\xbdL\xdc\xca<\x1e\xd7\xb...
3,114,1771560054124,1771560054124,Products that do not meet the criteria to be c...,44,Ice_cream,2251114,1771672495232,1771672495232,114,114,None,b'\xeb\xf6E=~\x14\x95\xbd\xc8I\xae<\xae\xf1\xa...
4,125,1771560054124,1771560054124,There is no evidence to support these legends....,56,Ice_cream,2668125,1771677928421,1771677928421,125,125,None,b'\x1b\x1c\xcd\xbc\xd1V\x1c=\xd6(I\xbd\n\x99\x...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,413,1771560054695,1771560054695,Hamburgers and veggie burgers served with chip...,40,Hamburger,574413,1771650665630,1771650665630,413,413,None,b'\xd1\xc0\x7f;E\x93\x0e\xbd\x8ce\xc4\xbb\xc1\...
496,10,1771560053971,1771560053971,"Afterwards, the ocean's current name was coine...",43,Pacific_Ocean,426010,1771648741880,1771648741880,10,10,None,b'\xe5C\xde\xbb\xe4\xb30=\xc2\x9eE\xbd\xb9\xb9...
497,479,1771560055036,1771560055036,"After Luna 24 in 1976, there were no soft land...",50,Moon_landing,212479,1771645987228,1771645987228,479,479,None,b'\xdf\x81\x96=\x84\xf8K\xbd:\x96\xae=\xac\x0b...
498,851,1771560055487,1771560055487,The Himalayas form most of the south-west port...,37,Himalayas,333851,1771647538204,1771647538204,851,851,None,b'\xf1\xe3a=\x85\x9cP\xbdXUA\xbd\x15M\xe5\xbbg...


In [19]:
conn = sqlite3.Connection('data.db')
cur = conn.cursor()
cur.execute('update embeddings set sentence_id = summary_id')
conn.commit()

In [10]:
import pandas as pd
import sqlite3

df = pd.read_sql('''
    select 
        unique_sentences.article_id, 
        link_count
    from (
        select distinct article_id
        from sentences
    ) as unique_sentences               
    join (
        select 
            article_id, 
            count(*) as link_count
        from (
            select distinct parent_article_id, article_id
            from links
        )
        group by article_id
    ) as link_counts
        on unique_sentences.article_id == link_counts.article_id
    left join summaries
        on unique_sentences.article_id == summaries.article_id
    where summaries.id is null

''', sqlite3.Connection('data.db'))

In [ ]:
import pandas as pd
import sqlite3

df = pd.read_sql('''
        select id, article_id
        from (
                select sentence_id from embeddings order by random() limit 1000) as embeddings
        join sentences on sentences.id == embeddings.sentence_id
''', sqlite3.Connection('data.db'))

df.sample(5)

,id,article_id
651,683,Eiffel_Tower
838,469,Taco
692,666,Eiffel_Tower
833,936,Himalayas
126,963,Sushi


In [7]:
df.sample()

,id,article_id
237,380023,American_Revolutionary_War


In [2]:
import pandas as pd
import sqlite3

pd.read_sql('''
		select 
			sentence_id as chunk_id, 
			embedding as vector
		from embeddings
		join sentences
			on embeddings.sentence_id == sentences.id
		where sentences.article_id = "Taco" and sentences.id = 441
''', sqlite3.Connection('data.db'))

,chunk_id,vector
0,441,b'\xb5H\x8b<\xb4i\xe9<!14\xbd\xf5\x8f]=\x1c^\x...
1,441,b'\xb5H\x8b<\xb4i\xe9<!14\xbd\xf5\x8f]=\x1c^\x...
2,441,b'\xb5H\x8b<\xb4i\xe9<!14\xbd\xf5\x8f]=\x1c^\x...
3,441,b'\xb5H\x8b<\xb4i\xe9<!14\xbd\xf5\x8f]=\x1c^\x...
4,441,b'\xb5H\x8b<\xb4i\xe9<!14\xbd\xf5\x8f]=\x1c^\x...
...,...,...
3302,441,b'\xb5H\x8b<\xb4i\xe9<!14\xbd\xf5\x8f]=\x1c^\x...
3303,441,b'\xb5H\x8b<\xb4i\xe9<!14\xbd\xf5\x8f]=\x1c^\x...
3304,441,b'\xb5H\x8b<\xb4i\xe9<!14\xbd\xf5\x8f]=\x1c^\x...
3305,441,b'\xb5H\x8b<\xb4i\xe9<!14\xbd\xf5\x8f]=\x1c^\x...


In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.Connection('data.db')
cur = conn.cursor()

cur.execute('VACUUM ')
# cur.execute("PRAGMA wal_checkpoint(TRUNCATE);")

conn.commit()

In [5]:
import pandas as pd
import sqlite3

pd.read_sql('''
		select 
			article_id, 
			title, 
			clean_title,
			count
		from suggestions
''', sqlite3.Connection('data.db'))

,article_id,title,clean_title,count
0,&,Ampersand,ampersand,3
1,'Ala'_al-Din_al-Bukhari,'Ala' al-Din al-Bukhari,'ala' al-din al-bukhari,44
2,'Round_About_Midnight,'Round About Midnight,'round about midnight,4
3,'Til_There_Was_You,'Til There Was You,'til there was you,3
4,(Everything_I_Do)_I_Do_It_for_You,(Everything I Do) I Do It for You,(everything i do) i do it for you,160
...,...,...,...,...
29061,Ōkōchi_Masatada,Ōkōchi Masatada,Ōkōchi masatada,1
29062,Śrauta,Śrauta,Śrauta,694
29063,Śuddhodana,Śuddhodana,Śuddhodana,612
29064,Śāstra_pramāṇam_in_Hinduism,Śāstra pramāṇam,Śāstra pramāṇam,338


In [34]:
df.iloc[0]['sentence']

"After Renaissance, the power of the Pope to serve as an effective arbitrator was questioned, despite brief efforts to restore the same. While the Pope was still part of discussions in the 17th century CE, the effectiveness of Pope's arbitration declined."

In [14]:
import pandas as pd
import sqlite3

# 2,186,104 sentences
# 30K articles
# 72 sentences per article on average
df = pd.read_sql('''

    select *
    from sentences 
    order by random()
    limit 10000
    
''', sqlite3.Connection('data.db'))

In [28]:
samp = df.sample().iloc[0]
print(samp['article_id'])
print()
sen = samp['sentence'].split()
s = 0
while s + 15 < len(sen):
    print(' '.join(sen[s : s + 15]))
    s = s + 15
print(' '.join(sen[s :]))

Rutgers_University

In 1925, the mascot was changed to Chanticleer, a fighting rooster from the medieval fable
Reynard the Fox (Le Roman de Renart) which was used by Geoffrey Chaucer in the
Canterbury Tales.


In [2]:
import requests
from bs4 import BeautifulSoup

In [ ]:
def get_soup(href, session, sem):
    headers = {'User-Agent': 'MyApp/1.0 (you@example.com)'}
    url = f'https://en.wikipedia.org/w/rest.php/v1/page/{href}/html'
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, features="html.parser")
    print(f'Got soup for: {href}')
    return soup



In [7]:
soup = get_soup('Tom_Cruise', None, None)

Got soup for: Tom_Cruise


In [13]:
for elem in soup.find_all('img'):
    print(elem.get('src'))

//upload.wikimedia.org/wikipedia/commons/thumb/b/b5/Tom_Cruise-2428.jpg/250px-Tom_Cruise-2428.jpg
//upload.wikimedia.org/wikipedia/commons/thumb/b/b2/Tom_Cruise_signature.svg/250px-Tom_Cruise_signature.svg.png
//upload.wikimedia.org/wikipedia/commons/thumb/d/db/1985_Tom_Cruise.jpg/250px-1985_Tom_Cruise.jpg
//upload.wikimedia.org/wikipedia/commons/thumb/a/a5/Tom_cruise_1989.jpg/250px-Tom_cruise_1989.jpg
//upload.wikimedia.org/wikipedia/commons/thumb/f/fa/TomCruiseOct07.jpg/250px-TomCruiseOct07.jpg
//upload.wikimedia.org/wikipedia/commons/thumb/2/24/Tom_Cruise_by_Gage_Skidmore.jpg/250px-Tom_Cruise_by_Gage_Skidmore.jpg
//upload.wikimedia.org/wikipedia/commons/thumb/1/19/Tom_Cruise_Sydney_Premiere_Mission_Impossible_part_one_%2853019353900%29_%28cropped%29.jpg/250px-Tom_Cruise_Sydney_Premiere_Mission_Impossible_part_one_%2853019353900%29_%28cropped%29.jpg
//upload.wikimedia.org/wikipedia/commons/thumb/3/33/Tom_Cruise_by_Gage_Skidmore_2.jpg/250px-Tom_Cruise_by_Gage_Skidmore_2.jpg
//upload.w

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.Connection('data.db')
cur = conn.cursor()

In [3]:
t = pd.read_sql(f'''
       select * from suggestions where count > 1000 order by random() limit 1
''', conn).iloc[0]['article_id']

with open('../backend/target.txt', 'wt') as file:
    file.write(t)

In [2]:
cur.execute('delete from embeddings')
conn.commit()

In [ ]:
s
t

,id,created,modified,sentence_id,summary_id,embedding
0,1,1771172757248,1771172757248,None,359,b'\x80\x04\x95\x8d\x06\x00\x00\x00\x00\x00\x00...


In [5]:
import numpy as np

np.frombytes(t.iloc[0]['embedding'])

AttributeError: module 'numpy' has no attribute 'frombytes'

In [ ]:
# t['embedding'] = t['embedding'].apply(lambda x: pickle.loads(x))

In [7]:
import numpy as np 
t['embedding'] = t['embedding'].apply(lambda x: array_to_blob(x))

In [11]:
data = [(x['id'], x['embedding']) for _, x in t[['id', 'embedding']].iterrows()]

In [12]:
cur.executemany(
    'update embeddings set embedding = ? where id = ?',
    data
)
conn.commit()

In [3]:
t = pd.read_sql(f'''
		select 
			* from embeddings limit 2
''', conn)
t['embedding'] = t['embedding'].apply(lambda x: pickle.loads(x))

In [6]:
t.iloc[0]['embedding'].dtype, t.iloc[0]['embedding'].shape

(dtype('float32'), (384,))

In [4]:
t = pd.read_sql(f'''
		select 
			article_id, 
			title, 
			title as clean_title,
			count(links.to_link) as count
		from summaries
		join links
			on links.to_link == summaries.article_id 
		group by article_id, title
            order by random()
            limit 100
''', conn)
t

,article_id,title,clean_title,count
0,Culture_industry,Culture industry,Culture industry,53
1,Pierre_de_Bérulle,Pierre de Bérulle,Pierre de Bérulle,105
2,Indian_country,Indian country,Indian country,90
3,Party_divisions_of_United_States_Congresses,Party divisions of United States Congresses,Party divisions of United States Congresses,48
4,Dan_Wilson_(musician),Dan Wilson (musician),Dan Wilson (musician),95
...,...,...,...,...
95,Cuban_dissident_movement,Cuban dissident movement,Cuban dissident movement,42
96,Roman_Syria,Roman Syria,Roman Syria,46
97,African_diaspora_religions,African diaspora religions,African diaspora religions,113
98,Old_Style_and_New_Style_dates,Old Style and New Style dates,Old Style and New Style dates,157


In [4]:
t = pd.read_sql(f'''
select *
      from summaries left join thumbnails using (article_id) where thumbnails.id is null
''', conn)
t

,id,created,modified,title,description,extract,article_id,id,created,modified,src,width,height
0,2,1771167529992,1771167529992,Libertarian socialism,Political philosophy,Libertarian socialism is an anti-authoritarian...,Libertarian_socialism,None,None,None,None,None,None
1,9,1771167530626,1771167530626,Anarchism,Political philosophy and movement,Anarchism is a political philosophy and moveme...,Anarchism,None,None,None,None,None,None
2,31,1771167532521,1771167532521,Family values,Cultural values based on traditional family st...,"Family values, sometimes referred to as famili...",Family_values,None,None,None,None,None,None
3,33,1771167532707,1771167532707,Critique of political economy,Social critique,Critique of political economy or simply the fi...,Critique_of_political_economy,None,None,None,None,None,None
4,45,1771167533771,1771167533771,Truman Doctrine,Anti-Soviet U.S. Cold War foreign policy,The Truman Doctrine is a U.S. foreign policy t...,Truman_Doctrine,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6252,26393,1771172425401,1771172425401,Save the Children,Non-government organization founded in 1919,"The Save the Children Fund, commonly known as ...",Save_the_Children,None,None,None,None,None,None
6253,26404,1771172427234,1771172427234,African Americans in Tennessee,Largest racial and ethnic minority in Tennesse...,African Americans are the second largest censu...,African_Americans_in_Tennessee,None,None,None,None,None,None
6254,26406,1771172427509,1771172427509,Insurgency in Balochistan,Insurgency in Iran and Pakistan,Baloch separatists and various Islamist milita...,Insurgency_in_Balochistan,None,None,None,None,None,None
6255,26409,1771172427967,1771172427967,Verstehen,Social science conception of understanding and...,"\nVerstehen, in the context of German philosop...",Verstehen,None,None,None,None,None,None


In [18]:
article_id = 'Libertarian_socialism'
import requests
headers = {'User-Agent': 'MyApp/1.0 (you@example.com)'}
url = f'https://en.wikipedia.org/api/rest_v1/page/summary/{article_id}'

r = requests.get(url, headers=headers)
r.raise_for_status()
r.json()

{'type': 'standard',
 'title': 'Libertarian socialism',
 'displaytitle': '<span class="mw-page-title-main">Libertarian socialism</span>',
 'namespace': {'id': 0, 'text': ''},
 'wikibase_item': 'Q1852084',
 'titles': {'canonical': 'Libertarian_socialism',
  'normalized': 'Libertarian socialism',
  'display': '<span class="mw-page-title-main">Libertarian socialism</span>'},
 'pageid': 18048,
 'lang': 'en',
 'dir': 'ltr',
 'revision': '1336912131',
 'tid': 'ebe2bdfc-034f-11f1-bff0-301a39a12ee1',
 'timestamp': '2026-02-06T11:35:22Z',
 'description': 'Political philosophy',
 'description_source': 'local',
 'content_urls': {'desktop': {'page': 'https://en.wikipedia.org/wiki/Libertarian_socialism',
   'revisions': 'https://en.wikipedia.org/wiki/Libertarian_socialism?action=history',
   'edit': 'https://en.wikipedia.org/wiki/Libertarian_socialism?action=edit',
   'talk': 'https://en.wikipedia.org/wiki/Talk:Libertarian_socialism'},
  'mobile': {'page': 'https://en.wikipedia.org/wiki/Libertarian

In [9]:
table = 'summaries'
t = pd.read_sql(f'''
select links.to_link, count(*)
            from links 
            left join {table}
                on {table}.article_id == links.to_link
            where {table}.article_id is null
            group by links.to_link
            order by count(*) desc
            limit 100
''', conn)
# print(t['title'])
# print(t['extract'])
t

,to_link,count(*)
0,Irene_Cara,183
1,R._Kelly,182
2,Ceuta,181
3,René_Girard,164
4,Salmon_P._Chase,164
...,...,...
95,1976_Republican_Party_presidential_primaries,155
96,1988_Republican_National_Convention,155
97,1990_FIFA_World_Cup,155
98,Actinide,155


In [122]:
t = pd.read_sql('''
    select *
    from summaries
    join embeddings
        on summaries.id == embeddings.summary_id
    order by random()
    limit 1
''', conn).iloc[0]
# print(t['title'])
# print(t['extract'])

In [123]:
tvec = pickle.loads(t['embedding'])
tvec

array([ 2.02816390e-02, -9.45562944e-02,  6.61663478e-04,  8.30215495e-03,
        2.61283219e-02,  2.79981550e-02, -5.89590892e-02, -2.63574030e-02,
       -2.68014404e-03,  8.29881709e-03, -5.40959425e-02,  5.22493310e-02,
       -4.90091257e-02,  9.98159721e-02,  3.83512713e-02,  1.95561498e-02,
       -2.11518854e-02,  3.80759202e-02,  1.64228119e-02, -4.19834964e-02,
        5.44787534e-02,  4.75984663e-02, -1.40313413e-02,  6.10070415e-02,
        6.16732053e-02, -6.23113327e-02,  2.58560386e-02, -1.16589488e-02,
       -3.24173388e-03, -6.55190945e-02, -2.73302756e-02, -3.57118063e-02,
        5.14993407e-02, -4.79396135e-02, -1.67671982e-02, -7.11428002e-02,
       -3.28804441e-02, -5.60915610e-03,  1.01110870e-02,  5.95986331e-03,
       -9.76122357e-03,  1.66929848e-02, -4.51576401e-04,  1.66278786e-03,
       -2.79408004e-02,  2.82687657e-02, -2.72699855e-02,  1.65593170e-03,
        4.71234098e-02, -6.97898045e-02,  1.99317955e-03,  1.90988439e-03,
        2.97724307e-02,  

In [212]:
t = pd.read_sql('''
    select *
    from summaries
    join embeddings
        on summaries.id == embeddings.summary_id
    where title like '%tv%'
    order by random()
    limit 1
''', conn).iloc[0]
print(t['title'])
print(t['extract'])
from scipy.spatial.distance import cosine
print(cosine(tvec, pickle.loads(t['embedding'])))

Apple TV (streaming service)
Apple TV, formerly known as Apple TV+, is a subscription over-the-top streaming service owned by Apple. The service launched on November 1, 2019, and it offers a selection of original production film and television series called Apple Originals. Its programming arm is Apple Studios.
0.6698301434516907


In [ ]:
# cur.execute('delete from embeddings')
# conn.commit()

In [23]:
pd.read_sql('''
    select 
        sources.source_link, 
        case when summaries.id is not null then True else False end as has_summary,
        count(links.from_link)
    from sources 
    left join summaries 
        using (source_link)
    left join links
        on sources.source_link == links.from_link
    group by sources.source_link
    order by count(links.from_link) desc
    limit 100
''', conn)

,source_link,has_summary,count(links.from_link)
0,Conservatism,0,8070
1,Catholic_Church,0,4991
2,Joe_Biden,1,4629
3,Ronald_Reagan,1,4503
4,John_F._Kennedy,0,4450
...,...,...,...
95,Bangladesh,1,2535
96,Holy_See,0,2518
97,Korean_War,1,2512
98,Roy_Halladay,1,2507


In [25]:
headers = {'User-Agent': 'MyApp/1.0 (you@example.com)'}
source_link = 'Joe_Biden'
url = f'https://en.wikipedia.org/api/rest_v1/page/summary/{source_link}'

import requests 

r = requests.get(url, headers=headers)
r.raise_for_status()
r.json()

{'type': 'standard',
 'title': 'Joe Biden',
 'displaytitle': '<span class="mw-page-title-main">Joe Biden</span>',
 'namespace': {'id': 0, 'text': ''},
 'wikibase_item': 'Q6279',
 'titles': {'canonical': 'Joe_Biden',
  'normalized': 'Joe Biden',
  'display': '<span class="mw-page-title-main">Joe Biden</span>'},
 'pageid': 145422,
 'thumbnail': {'source': 'https://upload.wikimedia.org/wikipedia/commons/thumb/6/68/Joe_Biden_presidential_portrait.jpg/330px-Joe_Biden_presidential_portrait.jpg',
  'width': 330,
  'height': 413},
 'originalimage': {'source': 'https://upload.wikimedia.org/wikipedia/commons/6/68/Joe_Biden_presidential_portrait.jpg',
  'width': 2400,
  'height': 3000},
 'lang': 'en',
 'dir': 'ltr',
 'revision': '1338209404',
 'tid': '84fb6e87-091c-11f1-9d85-0582b5419161',
 'timestamp': '2026-02-13T20:42:32Z',
 'description': 'President of the United States from 2021 to 2025',
 'description_source': 'local',
 'content_urls': {'desktop': {'page': 'https://en.wikipedia.org/wiki/Joe

In [6]:
pd.read_sql('''
    select * from sources
''', conn)

,source_link,created,modified,is_partial
0,Key,1771107510404,1771107510404,0
1,Soup,1771107510418,1771107510418,0
2,Star_Trek,1771107510429,1771107510429,0
3,China,1771107510441,1771107510441,0
4,The_Office,1771107510453,1771107510453,0
...,...,...,...,...
4656,Sogou_Pinyin,1771117911580,1771117911580,0
4657,Friedrich_Engels,1771117911588,1771117911588,0
4658,Mel_Allen,1771117911596,1771117911596,0
4659,Double_play,1771117911605,1771117911605,0


In [4]:
pd.read_sql('''
    select links.to_link, count(*)
    from links 
    left join sources
        on sources.source_link == links.to_link
    where sources.source_link is null
    group by links.to_link
    order by count(*) desc
    limit 100
''', conn)

,to_link,count(*)
0,Wil_Myers,214
1,Tom_Tresh,201
2,Eric_Hinske,195
3,Puroik_language,195
4,Hruso_language,194
...,...,...
95,Taman_language_(Myanmar),172
96,Rich_Gedman,171
97,Tim_Belcher,171
98,1964_Major_League_Baseball_season,169


In [3]:
s = pd.read_sql('''
          
    select *
    from summaries
    order by random()
    limit 1000    
''', conn).sample().iloc[0]

print(s['extract'])
print(s['title'])

The national flag of Canada, popularly referred to as the Maple Leaf, consists of a red field with a white square at its centre in the ratio of 1∶2∶1, in which is featured one stylized, red, 11-pointed maple leaf charged in the centre. It is the first flag to have been adopted by both houses of Parliament and officially proclaimed by the Canadian monarch as the country's official national flag. The flag has become the predominant and most recognizable national symbol of Canada.
Flag of Canada


In [19]:
import pickle

s = pd.read_sql('''
          
    select *
    from embeddings 
''', conn)
s.embedding.apply(lambda x: pickle.loads(x))

0      [0.060689878, 0.05323639, 0.04234263, 0.075609...
1      [0.02588305, -0.0008011006, 0.053584088, 0.098...
2      [-0.023254413, 0.02933463, 0.03757674, 0.00355...
3      [0.015221468, 0.065185316, 0.04403959, 0.02383...
4      [-0.003316514, 0.07554803, -0.02295627, 0.0653...
                             ...                        
995    [-0.008153627, -0.0057223914, -0.0031914327, 0...
996    [-0.07534776, -0.0067520677, 0.015891118, -0.0...
997    [-0.014982634, 0.009760863, 0.02642908, 0.0065...
998    [-0.024550002, 0.022460693, -0.008100989, -0.0...
999    [-0.017618371, 0.051454928, -0.018885454, 0.02...
Name: embedding, Length: 1000, dtype: object